In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
EDA_TABLES_DIR = PROCESSED_DIR / "eda_tables"


In [2]:
df = pd.read_csv(RAW_DIR / "Airbnb_Scraped_Data.csv")

In [3]:
df.head()

,Name,city,Price,Superhost,is_fav,Rating,Reviews,URL
0,"2-Bedroom with Bath, Sleeps 6, Amstelveen, 75m2",Amsterdam,"ج.م 51,630",Yes,0,4.78,819,https://www.airbnb.com/rooms/33092943?search_m...
1,Cozy houseboat with parking in the center of A...,Amsterdam,"ج.م 65,466",Yes,1,4.97,322,https://www.airbnb.com/rooms/58087008775012407...
2,Cozy Studios in Heart of Pijp,Amsterdam,"ج.م 101,624",Yes,1,4.92,686,https://www.airbnb.com/rooms/9323377?search_mo...
3,"Numa | 1 Bedroom Apartment w/ Sofa Bed, Double...",Amsterdam,"ج.م 67,717",No,0,4.7,325,https://www.airbnb.com/rooms/11454693974405935...
4,Spacious Serviced Apartment with River View,Amsterdam,"ج.م 187,948",Yes,1,4.85,763,https://www.airbnb.com/rooms/42446371?search_m...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Name       750 non-null    str  
 1   city       750 non-null    str  
 2   Price      750 non-null    str  
 3   Superhost  750 non-null    str  
 4   is_fav     750 non-null    int64
 5   Rating     711 non-null    str  
 6   Reviews    750 non-null    int64
 7   URL        750 non-null    str  
dtypes: int64(2), str(6)
memory usage: 364.8 KB


In [5]:
df.shape

(750, 8)

In [6]:
df.isnull().sum()

Name          0
city          0
Price         0
Superhost     0
is_fav        0
Rating       39
Reviews       0
URL           0
dtype: int64

In [7]:
df["city"].value_counts()

city
Amsterdam     50
Athens        50
Barcelona     50
Berlin        50
Budapest      50
Lisbon        50
London        50
Paris         50
Rome          50
Vienna        50
Madrid        50
Milan         50
Prague        50
Dublin        50
Copenhagen    50
Name: count, dtype: int64

In [8]:
df["Rating"].value_counts().head(10)

Rating
5.0     169
New      51
4.94     31
4.95     31
4.91     31
4.92     30
4.93     29
4.96     29
4.97     27
4.98     27
Name: count, dtype: int64

In [9]:
print(df["Price"].str.replace(r"\d", "9", regex=True).value_counts())

Price
ج.م 99,999     543
ج.م 999,999    207
Name: count, dtype: int64


In [10]:
USD_RATE = 50.5   # عدد الجنيهات في الدولار الواحد

df["Price_total_USD"] = (df["Price"].str.replace(r"[^\d]", "", regex=True).astype(int) / USD_RATE).round(2)
df["Price_per_night_USD"] = (df["Price_total_USD"] / 5).round(2)

print(df[["Price", "Price_total_USD", "Price_per_night_USD"]].head())
print(df["Price_total_USD"].describe())

df = df.drop(columns=["Price"])
print(df.columns.tolist())

         Price  Price_total_USD  Price_per_night_USD
0   ج.م 51,630          1022.38               204.48
1   ج.م 65,466          1296.36               259.27
2  ج.م 101,624          2012.36               402.47
3   ج.م 67,717          1340.93               268.19
4  ج.م 187,948          3721.74               744.35
count     750.000000
mean     1679.626453
std      1004.420806
min       344.690000
25%      1011.922500
50%      1446.570000
75%      2083.487500
max      7471.520000
Name: Price_total_USD, dtype: float64
['Name', 'city', 'Superhost', 'is_fav', 'Rating', 'Reviews', 'URL', 'Price_total_USD', 'Price_per_night_USD']


In [11]:
df.head()

,Name,city,Superhost,is_fav,Rating,Reviews,URL,Price_total_USD,Price_per_night_USD
0,"2-Bedroom with Bath, Sleeps 6, Amstelveen, 75m2",Amsterdam,Yes,0,4.78,819,https://www.airbnb.com/rooms/33092943?search_m...,1022.38,204.48
1,Cozy houseboat with parking in the center of A...,Amsterdam,Yes,1,4.97,322,https://www.airbnb.com/rooms/58087008775012407...,1296.36,259.27
2,Cozy Studios in Heart of Pijp,Amsterdam,Yes,1,4.92,686,https://www.airbnb.com/rooms/9323377?search_mo...,2012.36,402.47
3,"Numa | 1 Bedroom Apartment w/ Sofa Bed, Double...",Amsterdam,No,0,4.7,325,https://www.airbnb.com/rooms/11454693974405935...,1340.93,268.19
4,Spacious Serviced Apartment with River View,Amsterdam,Yes,1,4.85,763,https://www.airbnb.com/rooms/42446371?search_m...,3721.74,744.35


In [12]:
print(df["Rating"].isna().sum())
print(df[df["Rating"] == "New"]["Reviews"].describe())
print(df[df["Rating"].isna()][["Reviews", "is_fav"]].head(10))

39
count    51.000000
mean      1.372549
std       0.527666
min       0.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       2.000000
Name: Reviews, dtype: float64
    Reviews  is_fav
7         0       0
12        0       0
14        0       0
16        0       0
30        0       0
39        0       0
43        0       0
63        0       0
65        0       0
70        0       0


In [13]:
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

print(df["Rating"].dtype)
print(df["Rating"].isna().sum())
print(df["Rating"].describe())

float64
90
count    660.000000
mean       4.908818
std        0.113602
min        4.000000
25%        4.870000
50%        4.940000
75%        5.000000
max        5.000000
Name: Rating, dtype: float64


In [14]:
df.isnull().sum()

Name                    0
city                    0
Superhost               0
is_fav                  0
Rating                 90
Reviews                 0
URL                     0
Price_total_USD         0
Price_per_night_USD     0
dtype: int64

In [15]:
print(df["Superhost"].value_counts())

Superhost
Yes    406
No     344
Name: count, dtype: int64


In [16]:
df["Superhost"] = df["Superhost"].map({"Yes": 1, "No": 0})
print(df["Superhost"].value_counts())

Superhost
1    406
0    344
Name: count, dtype: int64


In [17]:
print(df["URL"].duplicated().sum())
print(df.duplicated().sum())

0
0


In [18]:
room_id = df["URL"].str.extract(r"/rooms/(\d+)")[0]
print(room_id.isna().sum())
print(room_id.duplicated().sum())

0
61


In [19]:
room_id = df["URL"].str.extract(r"/rooms/(\d+)")[0]
dup_mask = room_id.duplicated(keep=False)

print(df[dup_mask].groupby("city").size())
print(df.assign(room_id=room_id)[dup_mask].sort_values("room_id")[["room_id", "city", "Price_total_USD", "Rating", "Reviews"]].head(10))

city
Amsterdam      2
Athens         6
Berlin         9
Budapest       4
Copenhagen     2
Dublin         6
Lisbon        14
London         2
Madrid        26
Milan          5
Paris          6
Rome          15
Vienna        20
dtype: int64
                 room_id       city  Price_total_USD  Rating  Reviews
256  1049253169661259295     Lisbon           771.76    4.90      134
284  1049253169661259295     Lisbon           771.76    4.90      134
40   1060874386375062195  Amsterdam          2089.90    4.95       44
47   1060874386375062195  Amsterdam          2089.90    4.95       44
265  1093697959172627298     Lisbon           834.16    4.84      147
285  1093697959172627298     Lisbon           834.16    4.84      147
275             11582570     Lisbon          1108.55    4.99      215
282             11582570     Lisbon          1108.55    4.99      215
403  1223667358229932304       Rome           632.44    4.86      118
432  1223667358229932304       Rome           632.44    4.86 

In [20]:
df["listing_id"] = df["URL"].str.extract(r"/rooms/(\d+)")[0]
df = df.drop_duplicates(subset="listing_id", keep="first").reset_index(drop=True)

print(df.shape)
print(df["city"].value_counts())

(689, 10)
city
Barcelona     50
Prague        50
Amsterdam     49
London        49
Copenhagen    49
Budapest      48
Athens        47
Paris         47
Milan         47
Dublin        47
Berlin        45
Lisbon        43
Rome          41
Vienna        40
Madrid        37
Name: count, dtype: int64


In [21]:
print(df["Price_per_night_USD"].describe())
print(df.groupby("city")["Price_per_night_USD"].median().sort_values())
print(df.nlargest(5, "Price_per_night_USD")[["Name", "city", "Price_per_night_USD", "Rating", "Reviews"]])

count     689.000000
mean      345.500334
std       204.599428
min        68.940000
25%       205.720000
50%       294.650000
75%       428.280000
max      1494.300000
Name: Price_per_night_USD, dtype: float64
city
Athens        139.81
Budapest      187.14
Lisbon        208.38
Prague        223.03
Berlin        231.04
Vienna        281.60
Milan         282.19
Rome          284.71
Madrid        288.69
Barcelona     334.37
London        422.85
Dublin        437.61
Copenhagen    484.28
Paris         503.03
Amsterdam     547.62
Name: Price_per_night_USD, dtype: float64
                                                  Name        city  \
47   NEW: Luxurious Watervilla | Familyfriendly | S...   Amsterdam   
688  Former Dukes Residence - Central Charm - 4BR 2...  Copenhagen   
13        Canal-side family home with garden · Jordaan   Amsterdam   
330     Soho Skyline Penthouse with Terrace, Lift + AC      London   
16                                  The wellness Lodge   Amsterdam   

     Pr

In [22]:
df.columns

Index(['Name', 'city', 'Superhost', 'is_fav', 'Rating', 'Reviews', 'URL',
       'Price_total_USD', 'Price_per_night_USD', 'listing_id'],
      dtype='str')

In [24]:
mask = df["Name"].str.contains(r"\b[3-9]\s*(?:BR|bed|bedroom)", case=False, regex=True)
print(mask.sum())
print(df[mask]["Name"].head(10).tolist())

45
['Charming Canal Home | Spacious 3BR in Amsterdam', 'Casa Grande Landsmeer · 5BR Home near Amsterdam', '3 BEDRM APP (90m2) with canalview near Vondelpark', '1930s Architect-Designed Gem | 3BR 110m² Athens', '3BR Home Theater Apt with Projector & JBL sound', 'Brand New Lux 3BR near Metro & Port | Huge Terrace', 'Cozy apartments Plaza España, 3 bedrooms', '3br apartment in Sagrada Familia', 'Central 130m², 3BR, 2Bath, Work & Family-Friendly', 'Kertész Apartments 3(5AC,3BathR,4BedR)']


In [25]:
pattern = r"\b[3-9][\s-]*(?:br|bedr\w*|bedroom\w*|bdr\w*)\b"
mask = df["Name"].str.contains(pattern, case=False, regex=True)

print(mask.sum())
print(df[mask]["Name"].head(10).tolist())

df = df[~mask].reset_index(drop=True)
print(df.shape)
print(df["city"].value_counts())

53
['Charming Canal Home | Spacious 3BR in Amsterdam', 'Casa Grande Landsmeer · 5BR Home near Amsterdam', '3 BEDRM APP (90m2) with canalview near Vondelpark', '1930s Architect-Designed Gem | 3BR 110m² Athens', '3BR Home Theater Apt with Projector & JBL sound', 'Exceptional Central Athens 3 Bdrm 2 Bath, Sleeps 9', 'Brand New Lux 3BR near Metro & Port | Huge Terrace', 'Cozy apartments Plaza España, 3 bedrooms', 'Near the beach, pool, 3-bedroom, 2-bathroom, AC', '3br apartment in Sagrada Familia']
(636, 10)
city
Barcelona     47
Amsterdam     46
Milan         46
Copenhagen    45
London        44
Athens        43
Berlin        43
Dublin        43
Budapest      42
Lisbon        42
Paris         42
Rome          41
Prague        39
Madrid        37
Vienna        36
Name: count, dtype: int64


In [26]:
print(df.info())
print(df.isna().sum())

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DIR / "Airbnb_Cleaned.csv", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 636 entries, 0 to 635
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Name                 636 non-null    str    
 1   city                 636 non-null    str    
 2   Superhost            636 non-null    int64  
 3   is_fav               636 non-null    int64  
 4   Rating               556 non-null    float64
 5   Reviews              636 non-null    int64  
 6   URL                  636 non-null    str    
 7   Price_total_USD      636 non-null    float64
 8   Price_per_night_USD  636 non-null    float64
 9   listing_id           636 non-null    str    
dtypes: float64(3), int64(3), str(4)
memory usage: 318.0 KB
None
Name                    0
city                    0
Superhost               0
is_fav                  0
Rating                 80
Reviews                 0
URL                     0
Price_total_USD         0
Price_per_night_USD     0
listing